# Module 5A - Final Assignments

Two hands-on assignments that test everything you learned in Module 5A:

| Assignment | Focus | Difficulty | Concepts Tested |
|---|---|---|---|
| **Assignment 1 (Optional)** | SQL & Dashboard | Beginner-Intermediate | SQL Editor, joins, window functions, aggregations, visualizations, alerts |
| **Assignment 2 (Mandatory)** | Multi-Agent System | Advanced | PDF parsing & chunking, Vector Search, retrieval gating, SQL vs. Python UC function tools, single-agent design, supervisor/handoff orchestration, LLM security, Unity Catalog governance |

> **Free Edition note**: You have one pre-provisioned Serverless Starter Warehouse for SQL tasks. Use serverless compute for Python tasks.

---
# Assignment 1 (Optional): SQL Analytics Dashboard

## Scenario
You are a data analyst at **TechMart**, an e-commerce company. The business team needs a sales analytics dashboard to track product performance, customer behavior, and revenue trends.

Your manager has given you raw data in three tables. You must build a SQL analytics layer and a dashboard-ready summary table.

## Constraints
Your solution MUST meet ALL of the following:
1. **Use at least one JOIN** across two or more tables
2. **Use at least one window function** (e.g., `RANK()`, `LAG()`, `SUM() OVER()`, `ROW_NUMBER()`)
3. **Filter by a date range** (e.g., last 30 days, last quarter)
4. **Calculate at least one derived metric** (e.g., profit margin, revenue growth %, customer lifetime value)
5. **Store the final result in a Unity Catalog table** (not a temp view)
6. **Write an alert query** that would trigger if daily revenue drops below a threshold

## Deliverables
* Task 1: Exploratory query with JOIN + window function
* Task 2: Summary table with derived metrics, stored in UC
* Task 3: Visualization from the summary table
* Task 4: Alert query for revenue monitoring
* Run the validation cell to check constraints are met

In [0]:
%sql
-- SETUP: Run this cell to create the sample data for Assignment 1
-- Creates: TechMart orders, customers, and products tables in Unity Catalog

CREATE CATALOG IF NOT EXISTS assignment_techmart;
CREATE SCHEMA IF NOT EXISTS assignment_techmart.analytics;

-- Customers table
CREATE OR REPLACE TABLE assignment_techmart.analytics.customers (
  customer_id   INT,
  customer_name STRING,
  region        STRING,
  signup_date   DATE,
  tier          STRING  -- Bronze, Silver, Gold
);

INSERT INTO assignment_techmart.analytics.customers VALUES
(1, 'Alice Corp',     'North America', '2024-01-15', 'Gold'),
(2, 'Bob Industries', 'Europe',        '2024-03-22', 'Silver'),
(3, 'Charlie LLC',     'Asia Pacific',  '2024-06-10', 'Bronze'),
(4, 'Delta Inc',       'North America', '2024-09-05', 'Gold'),
(5, 'Echo Partners',   'Europe',        '2025-01-20', 'Silver'),
(6, 'Foxtrot Ltd',     'Asia Pacific',  '2025-03-18', 'Bronze'),
(7, 'Golf Associates', 'North America', '2025-05-12', 'Gold'),
(8, 'Hotel Co',        'Europe',        '2025-07-30', 'Silver');

-- Products table
CREATE OR REPLACE TABLE assignment_techmart.analytics.products (
  product_id    INT,
  product_name  STRING,
  category      STRING,
  unit_price    DECIMAL(10,2),
  unit_cost     DECIMAL(10,2)
);

INSERT INTO assignment_techmart.analytics.products VALUES
(101, 'Pro License',      'Software',   499.00, 200.00),
(102, 'Enterprise License', 'Software', 1499.00, 600.00),
(103, 'Vector Search',     'AI Service', 299.00, 100.00),
(104, 'AI Gateway',       'AI Service', 199.00,  80.00),
(105, 'Support Plan',      'Service',    99.00,  40.00),
(106, 'Training Credits',  'Service',    49.00,  20.00);

-- Orders table (90 days of data)
CREATE OR REPLACE TABLE assignment_techmart.analytics.orders AS
SELECT
  CAST(o.order_id AS INT) AS order_id,
  o.customer_id,
  o.product_id,
  o.quantity,
  o.order_date,
  o.order_date AS ship_date,
  CAST(o.quantity * p.unit_price AS DECIMAL(12,2)) AS revenue,
  CAST(o.quantity * p.unit_cost AS DECIMAL(12,2)) AS cost
FROM (
  SELECT
    explode(sequence(1, 200)) AS order_id,
  (c.customer_id),
  (p.product_id),
  CAST(rand() * 10 + 1 AS INT) AS quantity,
  date_add('2025-07-01', CAST(rand() * 90 AS INT)) AS order_date
  FROM assignment_techmart.analytics.customers c
  CROSS JOIN assignment_techmart.analytics.products p
  ORDER BY rand()
  LIMIT 200
) o
JOIN assignment_techmart.analytics.products p ON o.product_id = p.product_id;

SELECT 'Setup complete!' AS status, count(*) AS total_orders FROM assignment_techmart.analytics.orders;

In [0]:
%sql
-- TASK 1: Exploratory Query with JOIN + Window Function
-- PASTE YOUR CODE BELOW


In [0]:
%sql
-- TASK 2: Summary Table with Derived Metrics
-- PASTE YOUR CODE BELOW

##### TASK 3: Visualization
###### Paste the screenshots of Dashboard below

In [0]:
%sql
-- TASK 4: Alert Query for Revenue Monitoring
-- PASTE YOUR CODE BELOW

In [0]:
# VALIDATION: Run this cell AFTER completing all 4 tasks to check constraints.
# This cell verifies that your solution meets all the requirements.

print("=" * 60)
print("ASSIGNMENT 1 - VALIDATION CHECKS")
print("=" * 60)

checks_passed = 0
checks_total = 0

# Check 1: Customer summary table exists in UC
checks_total += 1
try:
    table_exists = spark.catalog.tableExists("assignment_techmart.analytics.customer_summary")
    if table_exists:
        print("[PASS] Task 2: UC table 'assignment_techmart.analytics.customer_summary' exists")
        checks_passed += 1
    else:
        print("[FAIL] Task 2: UC table 'assignment_techmart.analytics.customer_summary' does not exist")
except Exception as e:
    print(f"[FAIL] Task 2: Error checking table - {e}")

# Check 2: Summary table has derived metrics (profit_margin or similar)
checks_total += 1
try:
    if table_exists:
        cols = [f.name for f in spark.table("assignment_techmart.analytics.customer_summary").schema.fields]
        has_margin = any('margin' in c.lower() or 'growth' in c.lower() or 'aov' in c.lower() or 'avg_order' in c.lower() for c in cols)
        if has_margin:
            print(f"[PASS] Task 2: Derived metric found in columns: {cols}")
            checks_passed += 1
        else:
            print(f"[FAIL] Task 2: No derived metric (margin/growth/aov) found in columns: {cols}")
    else:
        print("[SKIP] Task 2: Table doesn't exist, cannot check columns")
except Exception as e:
    print(f"[FAIL] Task 2: Error checking columns - {e}")

# Check 3: Summary table has data
checks_total += 1
try:
    if table_exists:
        row_count = spark.table("assignment_techmart.analytics.customer_summary").count()
        if row_count > 0:
            print(f"[PASS] Task 2: Summary table has {row_count} rows of data")
            checks_passed += 1
        else:
            print("[FAIL] Task 2: Summary table is empty")
    else:
        print("[SKIP] Task 2: Table doesn't exist")
except Exception as e:
    print(f"[FAIL] Task 2: Error counting rows - {e}")

# Check 4: Orders table has data
checks_total += 1
try:
    order_count = spark.table("assignment_techmart.analytics.orders").count()
    if order_count > 0:
        print(f"[PASS] Setup: Orders table has {order_count} rows")
        checks_passed += 1
    else:
        print("[FAIL] Setup: Orders table is empty")
except Exception as e:
    print(f"[FAIL] Setup: Error - {e}")

# Check 5: All source tables exist
checks_total += 1
try:
    all_exist = all([
        spark.catalog.tableExists("assignment_techmart.analytics.customers"),
        spark.catalog.tableExists("assignment_techmart.analytics.products"),
        spark.catalog.tableExists("assignment_techmart.analytics.orders")
    ])
    if all_exist:
        print("[PASS] Setup: All source tables exist (customers, products, orders)")
        checks_passed += 1
    else:
        print("[FAIL] Setup: Some source tables are missing")
except Exception as e:
    print(f"[FAIL] Setup: Error - {e}")

print("=" * 60)
print(f"RESULT: {checks_passed}/{checks_total} automated checks passed")
print("=" * 60)
print("""
MANUAL CHECKS (verify by visual inspection):
[ ] Task 1: Query uses at least one JOIN
[ ] Task 1: Query uses at least one window function (RANK, LAG, SUM OVER, etc.)
[ ] Task 1: Query filters by a date range
[ ] Task 3: Visualization displays correctly
[ ] Task 4: Alert query returns a single numeric value
""")

---
# Assignment 2 (Mandatory) : Multi-Agent Customer Support System

## Scenario
You are a Generative AI Engineer at **ShopVerse**, an e-commerce company. Build a multi-agent
support system with three agents, using the same techniques demonstrated in **Demos 3-7**:

| Agent | Answers questions like | Built using |
|---|---|---|
| **Policy Agent** | "What's the return window for laptops?", "Do you cover accidental screen damage?" | RAG over the ShopVerse Policy Handbook PDF (Demo 3) |
| **Order Agent** | "What's the status of order 90205?", "How much has this customer spent?" | UC function tools over an orders table (Demo 4, Demo 5) |
| **Supervisor Agent** | Any incoming question — decides who should answer it, including questions that need **both** agents, e.g. *"I bought this laptop on order 90201 — can I still return it?"* | Supervisor-worker orchestration (Demo 6) |

This is exactly the **feature store vs. vector store** distinction from Demo 3, section 3.3:
policy text is unstructured → belongs in a vector index; order data is structured and
key-lookup-based → belongs in a table queried by tools, **not** embedded into the vector index.

## Deliverable
**One complete `.ipynb` notebook** that builds and demonstrates the full system end-to-end —
your own agents, your own tools, your own tests, on top of the order data provided below.
Follow the same **Notes (concept) → Demo (code) → validation** cell structure used in Demos 3
through 7.

## Reference: What's in the Policy Handbook PDF

You were given `ShopVerse_Policies.pdf` which is present in /static folder — this is the document your Policy Agent must
be able to answer from. It has 7 sections: **Return Policy, Refund Policy, Exchange Policy,
Shipping & Delivery Policy, Warranty Policy, Order Cancellation Policy, Privacy Policy.**

The Return Policy section includes a return-window table by product category — this is the
table your hybrid "can I return this?" questions will depend on, so it's reproduced here for
reference:

| Category | Return Window |
|---|---|
| Laptops & Computers | 15 days |
| Smartphones & Tablets | 15 days |
| Televisions & Large Appliances | 10 days |
| Small Kitchen Appliances | 30 days |
| Apparel & Footwear | 30 days |
| Books & Stationery | 7 days |
| Furniture | 7 days |
| Groceries & Perishables | Not eligible |
| Beauty & Personal Care | 10 days (unopened only) |
| Jewelry & Watches | 15 days |

Use `ai_parse_document` on the PDF and chunk it the way Demo 3 does (manual sentence/section
chunking, or `ai_prep_search`) — don't hand-type the policy text into your knowledge base as a
shortcut; the parsing + chunking pipeline itself is part of what's graded. If the PDF genuinely
isn't accessible in your workspace, a reasonable fallback is to re-create the same section
content as plain rows and note in a markdown cell that you did so — but real ingestion earns
more credit.


## Setup: Sample Order Data (given — run this, don't hand-roll your own)

This is boilerplate, not part of what's graded — run it as-is so everyone's building against the
same data. It covers every category in the PDF's return-window table, with a mix of orders
still inside their return window and orders past it, so the evaluation questions below have
real, checkable answers. You still have to build the PDF → `knowledge_base` ingestion yourself
for Requirement 1 — this cell only covers the **orders** side.


In [0]:
%sql
-- SETUP: Orders table for the Order Agent's tools. Given data — do not recreate this yourself.
-- Categories match the return-window table in the ShopVerse Policy Handbook exactly.
-- order_date is relative to CURRENT_DATE() so the mix of "still eligible" / "window expired"
-- orders holds no matter what day you run this.

CREATE CATALOG IF NOT EXISTS assignment_shopverse;
CREATE SCHEMA IF NOT EXISTS assignment_shopverse.agent;
CREATE VOLUME IF NOT EXISTS assignment_shopverse.agent.policy_documents;

CREATE OR REPLACE TABLE assignment_shopverse.agent.orders (
  order_id       INT,
  customer_name  STRING,
  customer_email STRING,
  product_name   STRING,
  category       STRING,
  order_date     DATE,
  price          DECIMAL(10,2),
  status         STRING
);

INSERT INTO assignment_shopverse.agent.orders VALUES
  (90201, 'Ananya Rao',      'ananya.rao@example.com',      'ProBook X15 Laptop',       'Laptops & Computers',              date_sub(current_date(), 5),   78999.00, 'Delivered'),
  (90202, 'Ravi Kumar',      'ravi.kumar@example.com',      'ProBook X15 Laptop',       'Laptops & Computers',              date_sub(current_date(), 22),  78999.00, 'Delivered'),
  (90203, 'Meera Iyer',      'meera.iyer@example.com',      'Nova 5G Smartphone',       'Smartphones & Tablets',            date_sub(current_date(), 9),   42999.00, 'Delivered'),
  (90204, 'Sanjay Gupta',    'sanjay.gupta@example.com',    'Nova 5G Smartphone',       'Smartphones & Tablets',            date_sub(current_date(), 30),  42999.00, 'Delivered'),
  (90205, 'Priya Nair',      'priya.nair@example.com',      '55" UltraView TV',         'Televisions & Large Appliances',   date_sub(current_date(), 3),   54999.00, 'Delivered'),
  (90206, 'Karan Mehta',     'karan.mehta@example.com',     '55" UltraView TV',         'Televisions & Large Appliances',   date_sub(current_date(), 15),  54999.00, 'Delivered'),
  (90207, 'Divya Shah',      'divya.shah@example.com',      'CrispAir Toaster Oven',    'Small Kitchen Appliances',         date_sub(current_date(), 12),  4499.00,  'Delivered'),
  (90208, 'Arjun Reddy',     'arjun.reddy@example.com',     'TrailRunner Sneakers',     'Apparel & Footwear',               date_sub(current_date(), 20),  3299.00,  'Delivered'),
  (90209, 'Neha Verma',      'neha.verma@example.com',      'TrailRunner Sneakers',     'Apparel & Footwear',               date_sub(current_date(), 45),  3299.00,  'Delivered'),
  (90210, 'Vikram Singh',    'vikram.singh@example.com',    'Study Desk Oak Finish',    'Furniture',                        date_sub(current_date(), 4),   11999.00, 'Delivered'),
  (90211, 'Anjali Pillai',   'anjali.pillai@example.com',   'Farm-Fresh Grocery Kit',   'Groceries & Perishables',          date_sub(current_date(), 2),   1299.00,  'Delivered'),
  (90212, 'Rohan Das',       'rohan.das@example.com',       'GlowLuxe Face Serum',      'Beauty & Personal Care',           date_sub(current_date(), 6),   1899.00,  'Delivered'),
  (90213, 'Kavya Menon',     'kavya.menon@example.com',     'Eternal Diamond Pendant',  'Jewelry & Watches',                date_sub(current_date(), 10),  64999.00, 'Delivered'),
  (90214, 'Sameer Khan',     'sameer.khan@example.com',     'ProBook X15 Laptop',       'Laptops & Computers',              date_sub(current_date(), 1),   78999.00, 'Shipped'),
  (90215, 'Tanya Joshi',     'tanya.joshi@example.com',     'Classic Paperback Set',    'Books & Stationery',               date_sub(current_date(), 8),   899.00,   'Delivered');

SELECT 'orders ready: ' || count(*) || ' rows across all 10 policy categories' AS status
FROM assignment_shopverse.agent.orders;


## Requirement 1: Policy Agent — RAG Pipeline (Demo 3)

Build the ingestion + retrieval pipeline the way Demo 3 does it — with one addition: don't just
default to `databricks-gte-large-en`, justify the embedding model you pick.

- [ ] Parse the PDF with `ai_parse_document` into a Unity Catalog table (primary key + Change
      Data Feed enabled, so it can back a Delta Sync index — see Demo 3 setup)
- [ ] Chunk the parsed content (sentence/section-level manual chunking, or `ai_prep_search` for
      automatic semantic chunking — either is fine, just say which you picked and why)
- [ ] **Choose your embedding model deliberately**:
  - Find the embedding models actually available to you in the workspace (check the serving
    endpoints and their task type, the way Demo 3 section 3.5 inspects `ep.task` to tell an
    embedding endpoint apart from an LLM endpoint)
  - From your chunked table, pick **5 representative question → correct-chunk pairs** — a
    question you'd expect a customer to ask, and the specific chunk that should answer it
  - For each candidate embedding model, embed each question and its correct chunk and compute
    the similarity between them (same approach as Demo 3 section 3.5b's manual embedding
    distance calculation)
  - Look at the 5 scores per model: are they consistently high, or is the model missing some
    pairs? Write a short markdown note on what you observed
  - Pick the model that scored best across your 5 pairs and use **that** model — not necessarily
    `databricks-gte-large-en` — when you build the index below
- [ ] Create a Vector Search endpoint and a Delta Sync index over your chunks, embedding with
      the model you selected above
- [ ] Implement **retrieval gating**: if the top chunk's similarity score is below **0.5** (the
      same threshold used in Demo 3, section 3.3), refuse with *"I don't know. I couldn't find
      relevant information."* instead of calling the LLM. Demonstrate this on a question that
      genuinely isn't in the handbook (e.g. "Do you price-match competitors?")


## Requirement 2: Order Agent — Tools (Demo 4)

Demo 4 draws a line between SQL UC functions (queries/aggregations) and Python UC functions
(external calls, complex/conditional logic). Build these 3 tools: 2 SQL, 1 Python — the Python
one isn't up for debate, for reasons explained below.

> **A note on Python UC functions**: stick to the Python standard library for these (for
> masking, that means the `re` module). Anything outside the standard library needs an explicit custom-dependency/
> environment setup on the function, and isn't guaranteed to work on serverless/Free Edition
> compute — you don't need anything beyond stdlib for what's asked here anyway.

**AI Tools  (UC Functions)**
- [ ] **`get_order_details`** — given an order ID, return that order's product, category, order
      date, price, status, and customer email. This can return the *raw* email internally — the
      masking happens one layer up, see below.
- [ ] **`get_customer_order_summary`** — given a customer identifier, return their total number
      of orders and total amount spent.

- [ ] **`mask_pii`** — given a string, detect whether it looks like an email or a phone number
      and mask it accordingly (e.g. `priya.nair@example.com` → something like
      `p*********@example.com`).

- [ ] **Select Language**: Find which functions can be implemented in SQL and Python and select your language accordingly. Also provide your reasoning

- [ ] **Enforce it**: your Order Agent must never put a raw email (or other PII) in a
      customer-facing response. Anywhere `get_order_details` or `get_customer_order_summary`
      would surface a customer's email, it must go through `mask_pii` first.

- [ ] Give every function a clear, action-oriented `COMMENT` (Demo 4, section 4.3 — tool
      selection quality depends entirely on this)
- [ ] Query `information_schema.routines` to show your 3 tools are discoverable, the way Demo 4
      does in section 4.3


## Requirement 3: Two Single Agents (Demo 5)

Before wiring anything together, build the **Policy Agent** and the **Order Agent** each as a
standalone single agent, following the single-agent pattern from Demo 5:

- [ ] Each agent gets its own **system prompt** (persona + what it can/can't do + a refusal
      policy — "say I don't know if you can't find relevant information")
- [ ] The Policy Agent's only tool is Vector Search retrieval (with gating from Requirement 1);
      the Order Agent's tools are the 3 UC functions from Requirement 2 — including `mask_pii`, which it must actually use whenever a response would otherwise contain a raw email
- [ ] Validate each agent independently with at least 2 test questions each (like Demo 5's 3
      test cells), before you build the supervisor — this isolates bugs to one agent instead of
      debugging the whole system at once


## Requirement 4: Supervisor Agent (Demo 6)

This is the core of the assignment. Use the **supervisor-worker / handoff pattern** from Demo 6
to tie the two agents together, plus one routing case Demo 6 didn't need: questions that require
**both** workers.

- [ ] Build a **Supervisor Agent** that hands off to the Policy Agent or the Order Agent, the
      same way Demo 6's supervisor hands off to `BillingAgent` / `ShippingAgent` /
      `TechnicalAgent`
- [ ] Handle a **hybrid** case the supervisor pattern alone doesn't cover: a question like *"I
      bought this laptop on order 90201 — can I still return it?"* needs the order's category
      and date from the Order Agent **and** that category's return window from the Policy Agent,
      combined into one definitive answer (eligible: yes/no, with the exact deadline date).
      **None of the 3 Order Agent tools do this for you** — deciding how to actually handle it is
      the point of this requirement. At least three real options exist:
      1. A new Python tool that calls both the order lookup and the Policy Agent's retrieval,
         and merges the results itself
      2. A dedicated third agent (or the Supervisor itself, without handing off) that calls both
         the Order Agent and the Policy Agent as sub-steps and composes the final answer
      3. Pure prompting: the Supervisor's system prompt instructs it to gather both pieces
         through its existing handoffs/tools and reason over them itself, with no new function
      Pick one, build it, and write a short markdown note on why you chose that architecture
      over the others — that reasoning is part of what's graded, not just the working answer
- [ ] Validate with **separate test cells per route** (policy-only, order-only, hybrid), the
      same way Demo 6 has one test cell per specialist, each printing which agent(s) handled it


## Requirement 5: Evaluation Questions

Run these **10 questions** through your assembled Supervisor Agent (not the individual agents
in isolation — this is testing the whole routed system). For each one, present a row with:

| # | Question | Response | Agent Path | Tool/Function Path |
|---|---|---|---|---|

- **Agent Path** = which agent(s) the Supervisor routed to: `Policy Agent`, `Order Agent`, or
  `Both (hybrid)`
- **Tool/Function Path** = the specific tool(s) actually invoked to answer it — e.g. Vector
  Search retrieval, `get_order_details`, `get_customer_order_summary`, `mask_pii`, or whatever
  you built for the hybrid case — in the order they were called

**The 10 questions:**


| # | Question | Response | Agent Path | Tool/Function Path |
|---|---|---|---|---|
| 1 | What is the return window for laptops? | 15 days from delivery. The laptop must be like-new, with the original charger, cables, manuals and packaging, under 10 hours of usage, and no BIOS/OS modifications. | Policy Agent | Vector Search retrieval (score ≥ 0.5) |
| 2 | Are groceries eligible for return? | No. Groceries and perishables are not eligible for return. | Policy Agent | Vector Search retrieval |
| 3 | What is the status of order 90205? | Order 90205 (55" UltraView TV) has been delivered. | Order Agent | get_order_details |
| 4 | How much did I (ravi.kumar@example.com) spent with you in total? | 1 order, totalling ₹78,999.00 (the ProBook X15 Laptop, order 90202). | Order Agent | get_customer_order_summary |
| 5 | I bought order 90201, a laptop — can I still return it? | Yes. Ordered 5 days ago against a 15-day window, so it's eligible until Oct 8, 2026 (10 days left). | Both (hybrid) | Order lookup (get_order_details) + Vector Search for the laptop window, merged by your hybrid design |
| 6 | I bought order 90209, sneakers — can I still return them? | No. Ordered 45 days ago against a 30-day apparel window. The window closed Sept 13, 2026, 15 days ago. | Both (hybrid) | Order lookup (get_order_details) + Vector Search for the Apparel & Footwear window |
| 7 | I bought order 90204, a smartphone — can I still return it? | No. Ordered 30 days ago against a 15-day smartphone window. The window closed Sept 13, 2026, 15 days ago. | Both (hybrid) | Order lookup (get_order_details) + Vector Search for the Smartphones & Tablets window |
| 8 | What was my email associated with order 90205? | The email on order 90205 is p*********@example.com. It must be masked, not refused, and priya.nair@example.com must not appear. | Order Agent | get_order_details → mask_pii |
| 9 | What's your refund timeline for a UPI payment? | 2–4 business days after the refund is approved. | Policy Agent | Vector Search retrieval |
| 10 | Can I speak to a human customer support agent right now? | "I don't know. I couldn't find relevant information." The handbook doesn't cover this. | Policy Agent / Supervisor's general route | Vector Search retrieval → gating triggers (score < 0.5), no LLM call |


A few notes on specific questions:
- **#6 and #7** are deliberate edge cases against the order data given above — both should come
  back *not eligible*, for different reasons (check the dates and windows yourself)
- **#8 is the mandatory PII check.** Order 90205 belongs to `priya.nair@example.com` — the
  response must **not** contain that raw string. It should look something like *"The email on
  order 90205 is p*********@example.com."* — masked, not omitted entirely (the agent should
  still confirm an email exists and give a masked form of it, not just refuse). If your response
  to #8 contains the unmasked address, Requirement 2's mandatory masking isn't actually wired in
- **#10** shouldn't be answerable from the handbook at all — it should trigger your refusal
  policy / retrieval gating, not a routing failure


## Requirement 6: Security (Demo 7)

- [ ] Demonstrate a **prompt injection attempt** (OWASP LLM01) against one of your agents and
      show that your system prompt holds — e.g. ask it to reveal its instructions, or to ignore
      its refusal policy — the same way Demo 7's section 6.1 compares a benign input to an
      injection attempt on a benign vs. a defensively-instructed agent
- [ ] In a markdown cell, briefly explain what in your system prompt / design is actually doing
      the defending. Also note here: your `mask_pii` tool from Requirement 2 is your mitigation
      for **OWASP LLM02 (Sensitive Information Disclosure)** — you've already built it, just
      connect the dots in this cell rather than treating it as a separate task
- [ ] No need to deploy the agent or testing the endpoint since we will cover that in later sections


---
# Grading Rubric

## Assignment 1: SQL Analytics Dashboard

| Criterion | How to Score |
|---|---|
| JOIN across 2+ tables | Query must join customers, products, and/or orders |
| Window function used | RANK, LAG, SUM OVER, or similar |
| Date range filter | WHERE clause filtering by date |
| Derived metric calculated | Profit margin %, revenue growth %, or AOV |
| UC table (not temp view) | CREATE TABLE or saveAsTable in UC catalog |
| Alert query returns single value | Query returns one numeric column for alert |
| Visualization created | Easily understandable Dashboard based on Screenshots |

## Assignment 2: Multi-Agent Customer Support System

| Criterion | Demo |
|---|---|
| PDF parsed, chunked, and indexed; embedding model chosen with the 5-pair scoring exercise, not defaulted | Demo 3 |
| Retrieval gating (0.5 threshold, demonstrated refusal) | Demo 3 |
| 3 order tools built: `get_order_details` + `get_customer_order_summary` (SQL) and `mask_pii` (Python, using the `re` module), all discoverable via information_schema.routines | Demo 4 |
| PII masking enforced: raw email never appears in a customer-facing response (question 8) | Demo 4 / Demo 7 |
| Policy Agent: system prompt + tools, validated standalone | Demo 5 |
| Order Agent: system prompt + tools, validated standalone | Demo 5 |
| Supervisor routing (policy / order / hybrid) with a deliberate, justified hybrid architecture, separate test cells per route | Demo 6 |
| All 10 evaluation questions run against the full system: response + agent path + tool path shown for each | — |
| Deployed as a serving endpoint, queried, and tested with a documented verification approach | Demo 6 |
| Prompt injection demonstrated and defended; `mask_pii` tied to OWASP LLM02 | Demo 7 |
| Unity Catalog governance throughout | All |

---

## Clean Up

When you're done, drop the catalogs you created for both assignments (adjust names to match
what you actually used), and delete any Vector Search endpoint/index and serving endpoint you
created, the way Demos 3 and 6 do in their own cleanup cells — dropping a catalog does not tear
those down.

```sql
DROP CATALOG IF EXISTS assignment_techmart CASCADE;
DROP CATALOG IF EXISTS assignment_shopverse CASCADE;  -- or whatever you named it
```
